# Screenshot bento box

Composes four app screenshots into a 2 × 2 bento box on a transparent background: rounded
corners, thin glassy frames, and a label (A–D) in each tile's top-left corner. Tiles read left
to right, top to bottom. Each column is sized so both columns end at the same height, so the
screenshots keep their aspect ratio and are not cropped.

Also saves an unlabeled copy to `docs/images/` for the README.

Screenshots live in `analyses/data/screenshots/` (gitignored).

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "backend/app/configs/config.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SCREENSHOT_DIR = ROOT / "analyses/data/screenshots"
OUTPUT_DIR = ROOT / "analyses/results"

## 1. List the screenshots

In [ ]:
from PIL import Image

available = sorted(SCREENSHOT_DIR.glob("*.png"))
for index, path in enumerate(available):
    with Image.open(path) as image:
        width, height = image.size
    print(f"{index:>2}  {width:>5} × {height:<5}  {path.name}")

## 2. Choose four

Keep exactly four lines uncommented. Their order sets the labels: A, B, C, D.

In [ ]:
SELECTED = [
    # "Screenshot from 2026-09-27 22-31-29.png",
    "Screenshot from 2026-09-27 22-35-56.png",
    "Screenshot from 2026-09-27 22-37-39.png",
    "Screenshot from 2026-09-27 22-33-01.png",
    "Screenshot from 2026-09-27 22-38-56.png",
    # "Screenshot from 2026-09-27 22-39-57.png",
]

LABELS = ["A", "B", "C", "D"]
assert len(SELECTED) == len(LABELS), f"Choose exactly 4 screenshots, got {len(SELECTED)}."
screenshots = [Image.open(SCREENSHOT_DIR / name).convert("RGBA") for name in SELECTED]

## 3. Style

In [ ]:
WIDTH = 2400  # canvas width in pixels, margins included
MARGIN = 48  # transparent room around the grid for the shadows
GAP = 44  # space between tiles; wider than a label's radius so labels clear the neighbors
FRAME = 10  # glass frame thickness
RADIUS = 28  # outer corner radius of each tile
SHADOW_BLUR = 18
SHADOW_OFFSET = 8
SHADOW_ALPHA = 70
SUPERSAMPLE = 4  # antialiasing factor for the rounded shapes
LABEL_SIZE = 40  # label font size
LABEL_DIAMETER = 64  # label circle, centered on each tile's top-left corner
OUTPUT = OUTPUT_DIR / "screenshot_bento.png"
README_OUTPUT = ROOT / "docs/images/screenshot_bento.png"  # unlabeled copy for the README

## 4. Helpers

In [ ]:
from functools import cache

from matplotlib import font_manager
from PIL import ImageChops, ImageDraw, ImageFilter, ImageFont, ImageOps

Size = tuple[int, int]
Box = tuple[int, int, int, int]  # x, y, width, height
RGBA = tuple[int, int, int, int]


@cache
def corner(radius: int) -> Image.Image:
    """An antialiased top-left quarter circle, the only part of a mask worth supersampling."""
    s = SUPERSAMPLE
    patch = Image.new("L", (radius * s, radius * s), 0)
    ImageDraw.Draw(patch).ellipse((0, 0, 2 * radius * s - 1, 2 * radius * s - 1), fill=255)
    return patch.resize((radius, radius), Image.Resampling.LANCZOS)


def rounded_mask(size: Size, radius: int, inset: int = 0) -> Image.Image:
    """An antialiased rounded-rectangle alpha mask, optionally inset from the edges."""
    width, height = size
    mask = Image.new("L", size, 0)
    body = Image.new("L", (width - 2 * inset, height - 2 * inset), 255)
    r = max(radius - inset, 0)
    if r:
        patch = corner(r)
        bw, bh = body.size
        body.paste(patch, (0, 0))
        body.paste(patch.transpose(Image.Transpose.FLIP_LEFT_RIGHT), (bw - r, 0))
        body.paste(patch.transpose(Image.Transpose.FLIP_TOP_BOTTOM), (0, bh - r))
        body.paste(patch.transpose(Image.Transpose.ROTATE_180), (bw - r, bh - r))
    mask.paste(body, (inset, inset))
    return mask


def hairline(size: Size, radius: int, inset: int, color: RGBA) -> Image.Image:
    """A 1-pixel rounded outline just inside the `inset` edge."""
    ring = ImageChops.subtract(
        rounded_mask(size, radius, inset), rounded_mask(size, radius, inset + 1)
    )
    layer = Image.new("RGBA", size, color)
    layer.putalpha(ring.point([v * color[3] // 255 for v in range(256)]))
    return layer


def vertical_gradient(size: Size, top: RGBA, bottom: RGBA) -> Image.Image:
    """An RGBA image fading from the `top` color to the `bottom` color."""
    width, height = size
    column = Image.new("RGBA", (1, height))
    for y in range(height):
        t = y / max(height - 1, 1)
        column.putpixel((0, y), tuple(round(a + (b - a) * t) for a, b in zip(top, bottom)))
    return column.resize(size)


def column_widths(
    ratios_left: list[float], ratios_right: list[float], content_width: int
) -> tuple[int, int]:
    """Widths of the two columns so their stacked tiles end at the same height.

    A tile's height is its image width × the image's height/width ratio, plus the frame.
    """
    image_width = content_width - GAP - 4 * FRAME
    left, right = sum(ratios_left), sum(ratios_right)
    left_image = round(image_width * right / (left + right))
    return left_image + 2 * FRAME, image_width - left_image + 2 * FRAME


def layout(images: list[Image.Image], content_width: int) -> tuple[list[Box], int]:
    """Tile boxes (x, y, width, height), in A–D order, within the content area."""
    ratios = [image.height / image.width for image in images]
    columns = [[0, 2], [1, 3]]  # A over C, B over D
    widths = column_widths(
        [ratios[i] for i in columns[0]], [ratios[i] for i in columns[1]], content_width
    )
    boxes: list[Box] = [(0, 0, 0, 0)] * len(images)
    x = 0
    for column, width in zip(columns, widths):
        y = 0
        for index in column:
            height = round((width - 2 * FRAME) * ratios[index]) + 2 * FRAME
            boxes[index] = (x, y, width, height)
            y += height + GAP
        x += width + GAP
    # Rounding can leave the columns a pixel apart; stretch the shorter bottom tile.
    bottom = max(boxes[i][1] + boxes[i][3] for i in (2, 3))
    for i in (2, 3):
        bx, by, bw, _ = boxes[i]
        boxes[i] = (bx, by, bw, bottom - by)
    return boxes, bottom


def label_font(size: int) -> ImageFont.FreeTypeFont:
    path = font_manager.findfont(font_manager.FontProperties(family="DejaVu Sans", weight="bold"))
    return ImageFont.truetype(path, size)


def glass_tile(image: Image.Image, size: Size) -> Image.Image:
    """One tile: the screenshot inside a thin, rounded, semi-transparent glass frame."""
    width, height = size
    tile = Image.new("RGBA", size, (0, 0, 0, 0))

    # Frame: a frosted white ring, brighter at the top like light on glass.
    glass = vertical_gradient(size, (255, 255, 255, 120), (255, 255, 255, 55))
    tile.paste(glass, (0, 0), rounded_mask(size, RADIUS))

    # Screenshot, scaled to the opening; the rounding matches the frame's inner edge.
    inner = (width - 2 * FRAME, height - 2 * FRAME)
    fitted = ImageOps.fit(image, inner, Image.Resampling.LANCZOS)
    opening = rounded_mask(inner, RADIUS - FRAME)
    fitted.putalpha(ImageChops.darker(fitted.getchannel("A"), opening))
    tile.alpha_composite(fitted, (FRAME, FRAME))

    # Hairlines: a bright outer rim and a faint dark edge around the opening.
    tile.alpha_composite(hairline(size, RADIUS, 0, (255, 255, 255, 190)))
    tile.alpha_composite(hairline(size, RADIUS, FRAME, (0, 0, 0, 90)))

    return tile


def label_badge(label: str, font: ImageFont.FreeTypeFont) -> Image.Image:
    """A black circle with a white letter, drawn over a tile's top-left corner."""
    d = LABEL_DIAMETER
    s = SUPERSAMPLE
    badge = Image.new("RGBA", (d * s, d * s), (0, 0, 0, 0))
    ImageDraw.Draw(badge).ellipse((0, 0, d * s - 1, d * s - 1), fill=(0, 0, 0, 255))
    badge = badge.resize((d, d), Image.Resampling.LANCZOS)
    ImageDraw.Draw(badge).text(
        (d / 2, d / 2), label, font=font, fill=(255, 255, 255, 255), anchor="mm"
    )
    return badge


def shadows(size: Size, boxes: list[Box]) -> Image.Image:
    """Soft drop shadows for every tile, blurred together in one pass."""
    alpha = Image.new("L", size, 0)
    for x, y, width, height in boxes:
        alpha.paste(rounded_mask((width, height), RADIUS), (x, y + SHADOW_OFFSET))
    alpha = alpha.filter(ImageFilter.GaussianBlur(SHADOW_BLUR))
    layer = Image.new("RGBA", size, (0, 0, 0, 0))
    layer.putalpha(alpha.point([v * SHADOW_ALPHA // 255 for v in range(256)]))
    return layer

## 5. Compose and save

In [ ]:
assert GAP > LABEL_DIAMETER / 2 and MARGIN >= LABEL_DIAMETER / 2, (
    "GAP and MARGIN must exceed half of LABEL_DIAMETER, or labels overlap neighbors or get cut off."
)

boxes, content_height = layout(screenshots, WIDTH - 2 * MARGIN)
canvas_size = (WIDTH, content_height + 2 * MARGIN)
canvas = Image.new("RGBA", canvas_size, (0, 0, 0, 0))
font = label_font(LABEL_SIZE)

placed = [(x + MARGIN, y + MARGIN, w, h) for x, y, w, h in boxes]
canvas.alpha_composite(shadows(canvas_size, placed))
for image, (x, y, w, h) in zip(screenshots, placed):
    canvas.alpha_composite(glass_tile(image, (w, h)), (x, y))

README_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
canvas.save(README_OUTPUT, optimize=True)
print(f"{README_OUTPUT.relative_to(ROOT)}  {canvas.width} × {canvas.height}")

# Labels go on last so no neighboring tile covers the part hanging outside the corner.
for label, (x, y, _, _) in zip(LABELS, placed):
    offset = LABEL_DIAMETER // 2
    canvas.alpha_composite(label_badge(label, font), (x - offset, y - offset))

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
canvas.save(OUTPUT)
print(f"{OUTPUT.relative_to(ROOT)}  {canvas.width} × {canvas.height}")

Preview on a checkerboard so the transparency shows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(12, 12 * canvas.height / canvas.width))
checker = np.indices((canvas.height // 24 + 1, canvas.width // 24 + 1)).sum(axis=0) % 2
ax.imshow(
    np.kron(checker, np.ones((24, 24)))[: canvas.height, : canvas.width],
    cmap="Greys",
    vmin=-4,
    vmax=4,
)
ax.imshow(canvas)
ax.set_axis_off()
plt.show()